# Supplementary Figure 4 - Additional exemplar genes and gene-set trajectories

Sub-panels under `Fig1/supp_4/<panel>/`. **S4a** additional ASD-risk / top exemplar genes
(stacked-area UTR usage along pseudotime). **S4b** mean 3'UTR-length-index W(t) trajectories
for the curated functional gene sets (RNA processing / cognition / proteostasis).


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import rankdata

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S
S.apply_rcparams()

FIG = "Fig1/supp_4"
BASE = Path(f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling")
SMOOTH = BASE / "UTR_count_table_genelevel_harmony_proportions_lineage_smoothed.h5ad"
comp = pd.read_csv(BASE / "DirMult_optimized" / "composite_ranking_optimized2_df5.csv")
members = pd.read_csv(BASE / "DirMult_optimized" / "ORA" / "curated_geneset_members_fig1supp.csv")

print("loading smoothed proportions...")
adata = sc.read_h5ad(SMOOTH)
pt = adata.obs["dpt_pseudotime"].values
print("smoothed:", adata.shape)

def gene_props(gene):
    mask = (adata.var["gene_name"] == gene).values
    if mask.sum() == 0:
        return None, None
    sub = adata[:, mask]
    pr = sub.layers["smoothed_proportions"]
    pr = pr.toarray() if hasattr(pr, "toarray") else np.asarray(pr)
    return np.asarray(pr, float), sub.var["utr_size"].values.astype(float)

def bin_mean(x, vals, nb=40):
    ok = ~np.isnan(x)
    b = pd.cut(x[ok], bins=nb, labels=False)
    cen = np.array([x[ok][b == i].mean() for i in range(nb)])
    mu = np.array([np.nanmean(vals[ok][b == i]) for i in range(nb)])
    se = np.array([np.nanstd(vals[ok][b == i]) / max(np.sqrt((b == i).sum()), 1) for i in range(nb)])
    keep = ~np.isnan(cen)
    return cen[keep], mu[keep], se[keep]

In [ ]:
# S4a - additional exemplar genes (stacked-area UTR usage)
EXEMPLARS = ["Prodh", "Tcf4", "Bcl11a", "Nova2", "Dyrk1a", "Kmt2d", "Hnrnpr"]
fig, axes = plt.subplots(2, 4, figsize=(S.FULL_W_PX / 300, 1100 / 300))
axes = axes.ravel()
for ax, gene in zip(axes, EXEMPLARS):
    pr, sizes = gene_props(gene)
    row = comp[comp.gene_name == gene].iloc[0]
    order = np.argsort(sizes); pr = pr[:, order]; sizes = sizes[order]; K = pr.shape[1]
    colors = ["#E24A33", "#4878CF"][:K] if K <= 2 else list(sns.color_palette("RdYlBu_r", K))
    valid = ~np.isnan(pt)
    b = pd.cut(pt[valid], bins=40, labels=False)
    cen = np.array([pt[valid][b == i].mean() for i in range(40)])
    mp = np.array([pr[valid][b == i].mean(axis=0) for i in range(40)])
    keep = ~np.isnan(cen); cen, mp = cen[keep], mp[keep]
    mp = mp / mp.sum(axis=1, keepdims=True)
    ax.stackplot(cen, mp.T, colors=colors, alpha=0.85)
    ax.set_xlim(cen.min(), cen.max()); ax.set_ylim(0, 1)
    dcol = S.COL_SHORT if row.delta_psi < 0 else S.COL_LONG
    ax.set_title(f"$\\it{{{gene}}}$  K={K}\n$\\Delta\\psi$={row.delta_psi:+.2f}, padj={row.padj:.0e}",
                 fontsize=S.FONT_ANNOT - 0.5, color=dcol, pad=2)
    ax.set_xticks([]); ax.tick_params(labelsize=S.FONT_TICK - 1)
    if list(axes).index(ax) % 4 == 0:
        ax.set_ylabel("UTR-site\nproportion", fontsize=S.FONT_TICK)
axes[-1].axis("off")
axes[-1].text(0.1, 0.5, "stacked area = mean\nUTR-site proportions\nalong pseudotime\n(proximal warm ->\ndistal cool)",
              fontsize=S.FONT_ANNOT - 0.5, va="center", transform=axes[-1].transAxes)
fig.subplots_adjust(left=0.06, right=0.98, top=0.88, bottom=0.05, hspace=0.45, wspace=0.25)
S.save_and_show(fig, "fig_s4a_additional_exemplars", figure=FIG)

### S4a variant: equal-cell x-axis + cell-type boxes (mimics the Fig 1e uniform left subpanel)

Same additional exemplars but on the equal-cell (ECDF) pseudotime axis with a cell-type-majority
colour bar instead of pseudotime ticks, matching the main Fig 1e uniform panel. Saved as
`fig_s4a_additional_exemplars__equalcell_ctbar`.

In [ ]:
# --- S4a variant: equal-cell axis + cell-type boxes (mimics the Fig 1e uniform left subpanel) ---
from matplotlib.lines import Line2D as _L2D
_v = ~np.isnan(pt)
_u = np.full_like(pt, np.nan)
_u[_v] = (rankdata(pt[_v], method="average") - 0.5) / _v.sum()   # equal-cell (ECDF)
_ct = adata.obs["celltype"].astype(str).values
_CT = S.MOUSE_LINEAGE_ORDER
EXEMPLARS = ["Prodh", "Tcf4", "Bcl11a", "Nova2", "Dyrk1a", "Kmt2d", "Hnrnpr"]
fig, axes = plt.subplots(2, 4, figsize=(S.FULL_W_PX / 300, 1100 / 300))
axes = axes.ravel()
uv = _u[_v]; ctv = _ct[_v]; bb = pd.cut(uv, bins=40, labels=False)
for ax, gene in zip(axes, EXEMPLARS):
    pr, sizes = gene_props(gene)
    row = comp[comp.gene_name == gene].iloc[0]
    order = np.argsort(sizes); pr = pr[:, order]; sizes = sizes[order]; K = pr.shape[1]
    colors = ["#E24A33", "#4878CF"][:K] if K <= 2 else list(sns.color_palette("RdYlBu_r", K))
    prv = pr[_v]
    cen = np.array([uv[bb == i].mean() for i in range(40)])
    mp = np.array([prv[bb == i].mean(axis=0) for i in range(40)])
    keep = ~np.isnan(cen); cen, mp = cen[keep], mp[keep]
    mp = mp / mp.sum(axis=1, keepdims=True)
    ax.stackplot(cen, mp.T, colors=colors, alpha=0.85)
    ax.set_xlim(0, 1); ax.set_ylim(-0.14, 1.0)
    for ct in _CT:                                          # cell-type boxes on the equal-cell axis
        cu = uv[ctv == ct]
        if len(cu):
            ax.fill_between([cu.min(), cu.max()], -0.13, -0.02,
                            color=S.paper_palette_celltypes[ct], alpha=0.9, linewidth=0)
    ax.axhline(0, color="grey", lw=0.4, alpha=0.3)
    dcol = S.COL_SHORT if row.delta_psi < 0 else S.COL_LONG
    ax.set_title(f"$\\it{{{gene}}}$  K={K}\n$\\Delta\\psi$={row.delta_psi:+.2f}, padj={row.padj:.0e}",
                 fontsize=S.FONT_ANNOT - 0.5, color=dcol, pad=2)
    ax.set_xticks([]); ax.tick_params(axis="x", bottom=False, labelbottom=False)
    ax.spines["bottom"].set_visible(False)
    if list(axes).index(ax) % 4 == 0:
        ax.set_ylabel("UTR-site\nproportion", fontsize=S.FONT_TICK)
axes[-1].axis("off")
present = [c for c in _CT if c in set(_ct)]
ct_h = [_L2D([0], [0], marker="s", color="w", markerfacecolor=S.paper_palette_celltypes[c],
             markersize=5, label=c) for c in present]
axes[-1].legend(handles=ct_h, loc="center", fontsize=S.FONT_LEGEND - 0.5, frameon=False,
                title="cell type (axis bar)", title_fontsize=S.FONT_LEGEND - 0.5, labelspacing=0.3)
axes[-1].text(0.5, 0.02, "equal-cell (ECDF) pseudotime;\nstacked area = mean UTR-site\nproportions "
              "(proximal warm -> distal cool)", fontsize=S.FONT_ANNOT - 1, va="bottom", ha="center",
              transform=axes[-1].transAxes)
fig.subplots_adjust(left=0.06, right=0.98, top=0.88, bottom=0.05, hspace=0.45, wspace=0.25)
S.save_and_show(fig, "fig_s4a_additional_exemplars__equalcell_ctbar", figure=FIG)


In [ ]:
# S4b - gene-set mean W(t) trajectories (functional categories)
def cell_W_for_gene(pr, sizes):
    K = pr.shape[1]
    r = (rankdata(sizes, method="average") - 1) / (K - 1) if K > 1 else np.array([0.5])
    return pr @ r

CAT_COL = {"RNA processing": "#1f78b4", "Cognition/neural": "#33a02c", "Proteostasis": "#e31a1c"}
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 760 / 300))
for cat in ["RNA processing", "Cognition/neural", "Proteostasis"]:
    genes = members.loc[members.category == cat, "gene_name"].tolist()
    Ws = []
    for g in genes:
        pr, sizes = gene_props(g)
        if pr is None or pr.shape[1] < 2:
            continue
        Ws.append(cell_W_for_gene(pr, sizes))
    if not Ws:
        continue
    Wset = np.nanmean(np.vstack(Ws), axis=0)            # per-cell mean W over the set
    cen, mu, se = bin_mean(pt, Wset, nb=40)
    ax.plot(cen, mu, "-", color=CAT_COL[cat], lw=1.4, label=f"{cat} (n={len(Ws)})")
    ax.fill_between(cen, mu - se, mu + se, color=CAT_COL[cat], alpha=0.18, linewidth=0)
ax.set_xlabel("pseudotime", fontsize=S.FONT_LABEL)
ax.set_ylabel("mean 3$'$UTR-length index W(t)", fontsize=S.FONT_LABEL)
ax.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="best")
ax.set_title("functional gene-set 3'UTR-length trajectories", fontsize=S.FONT_ANNOT, pad=3)
ax.tick_params(labelsize=S.FONT_TICK); ax.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.13, right=0.96, top=0.91, bottom=0.14)
S.save_and_show(fig, "fig_s4b_geneset_trajectories", figure=FIG)
del adata